# LLM: records to a fine-tune

An `llm_finetune` dataset is records, not rows: `instruction` / `input` /
`output`, or a `messages` list for chat. This takes yours from a file, registers
it, and starts a fine-tune — **if this machine can run one**, which is the cell
most of this notebook is about. A fine-tune needs the `llm` extra and a GPU it
will actually fit on, and finding that out forty minutes in is the failure worth
avoiding.

In [ ]:
import orinth
from pathlib import Path

# A .jsonl of {"instruction", "input", "output"} or {"messages": [...]}.
SOURCE = Path("~/Downloads/my-instructions.jsonl").expanduser()
print(SOURCE, "exists:", SOURCE.exists())

In [ ]:
import json
import tempfile

if not SOURCE.exists():
    SOURCE = Path(tempfile.mkdtemp()) / "example-instructions.jsonl"
    EXAMPLES = [
        ("Summarise the finding in one sentence.", "Periapical radiolucency at tooth 36, 4mm.",
         "A 4mm periapical radiolucency is present at tooth 36."),
        ("Rewrite the note for a patient.", "Impacted 48 with pericoronitis.",
         "The lower right wisdom tooth is trapped and the gum around it is inflamed."),
        ("List the follow-up steps.", "Post-extraction, day 1.",
         "Rinse with warm salt water, avoid the socket when brushing, and review in one week."),
    ]
    with SOURCE.open("w") as handle:
        for index in range(24):
            instruction, source_text, output = EXAMPLES[index % len(EXAMPLES)]
            handle.write(json.dumps({
                "instruction": instruction,
                "input": f"{source_text} (case {index})",
                "output": output,
            }) + "\n")
    print("using the example instead:", SOURCE)

print(SOURCE.read_text().splitlines()[0])

## 1 · Upload and inspect

In [ ]:
dataset = orinth.datasets.upload(SOURCE, name="llm pipeline")
print(dataset.id, "·", dataset.task_type, "·", dataset.format)
print(dataset.splits, "|", dataset.readiness.state)

frame = orinth.datasets.load(dataset.id, "train", limit=200)
print(frame.columns)
frame.head(3)

## 2 · Length, before anything downloads

The one dataset property that decides the run's cost. A record longer than
`max_seq_length` is truncated *and still counted*, which teaches the model to
stop mid-sentence.

In [ ]:
import polars as pl

if "messages" in frame.columns:
    lengths = pl.DataFrame(
        [
            {"chars": sum(len(str(m.get("content", ""))) for m in (row["messages"] or []))}
            for row in frame.iter_rows(named=True)
        ]
    )
else:
    lengths = frame.select(
        (
            pl.col("instruction").fill_null("")
            + " " + pl.col("input").fill_null("")
            + " " + pl.col("output").fill_null("")
        ).str.len_chars().alias("chars")
    )

print(lengths.describe())
print(f"\n~{lengths['chars'].quantile(0.99) / 4:.0f} tokens at p99, taking 4 chars to the token")

## 3 · Can this machine run it?

`runnable: false` on every option means the `llm` extra is missing or there is no
GPU the runner will accept. That is a fact worth reading before starting a job,
not after.

In [ ]:
options = orinth.train.options("llm_finetune")
for entry in options:
    print(f"  {entry['id']:22} runnable={str(entry['runnable']):5} download={entry['needs_download']}  {entry['description'][:60]}")

compute = orinth.train.compute()
for framework in compute["frameworks"]:
    devices = ", ".join(device["name"] for device in framework["devices"]) or "cpu only"
    print(f"\n{framework['name']:12} accelerated={framework['accelerated']}  {devices}")

runnable = [entry for entry in options if entry["runnable"]]
print("\n", len(runnable), "runnable option(s) here")

## 4 · Start it

Only when something is runnable, and only on `START = True`. A fine-tune pulls a
multi-gigabyte base model and holds the GPU for the length of the run — not
something a template should trigger because you pressed Run all.

In [ ]:
START = False

if not runnable:
    print("Nothing runnable here. Install the LLM extra:")
    print("  cd backend && uv sync --extra llm")
    print("On Linux with CUDA, add: --extra llm-cuda")
elif not START:
    print("Ready. Set START = True to launch:")
    print(f"  orinth.train.start('{dataset.id}', model_option_id='{runnable[0]['id']}',")
    print("                     task_type='llm_finetune', epochs=1)")
else:
    run = orinth.train.start(
        dataset.id,
        model_option_id=runnable[0]["id"],
        task_type="llm_finetune",
        name="llm pipeline",
        epochs=1,
        hyperparameters={"max_seq_length": 1024},
    )
    print("queued", run.id, "— watch it on /training, or:")
    print("  final = orinth.train.wait_for(run.id)")

## After it lands

A finished fine-tune registers a model like any other, so the rest is the same
vocabulary as every other pipeline:

```python
final = orinth.train.wait_for(run.id)
orinth.models.get(final.model_id)
orinth.inference.predict(final.model_id, text="Summarise the finding in one sentence.")
```

For an interactive session, export to GGUF and serve it from the Models page —
chat streams token by token there, which a notebook cell does not.